01_data_preparation_eda.ipynb

Step 1 to Step 8

**STEP 1: BUSINESS UNDERSTANDING**



#### Objective

The objective of this project is to predict whether a customer will churn during a renewal cycle using historical customer interaction data.

#### Problem Statement

The business aims to identify customers who are likely to churn at least 45 days before their renewal date, enabling early intervention strategies.

#### Unit of Analysis

Each row represents:
    1 Customer × 1 Renewal Cycle

#### Target Variable

* Churn = 1 → Customer did not renew
* Churn = 0 → Customer renewed

#### Time Anchor

    Prospect_Renewal_Date


#### Modeling Approach

##### Definition

Only data available before 45 days of the renewal date is used.

##### Time Window

Call_Date <= (Renewal_Date - 45 days)

#### Purpose

* Avoid data leakage
* Mimic real-world prediction
* Enable early churn detection

#### Business Value

Helps the business take proactive retention actions well before renewal.



**STEP 2: Data Understanding**

In this step, we explore all the provided datasets to understand their structure, content, and relationships.

The objectives of this step are:

* To examine dataset dimensions and structure
* To understand column types and meanings
* To identify missing values and inconsistencies
* To detect duplicates
* To identify key columns for joining datasets

The datasets used in this project are:

1. **Billings Dataset** – Contains customer information, renewal details, and target variable
2. **Renewal Calls Dataset** – Contains call interactions during the renewal process
3. **Emails Dataset** – Contains CRM/email interaction signals
4. **Customer Care Calls Dataset** – Contains support and complaint-related interactions

This step helps build a strong understanding of the data before proceeding to cleaning and feature engineering.


In [271]:
!pip install pandas

1. Import libraries

In [272]:
import pandas as pd
import numpy as np

2. Load datasets

In [273]:
billings = pd.read_csv(r"D:\INT230\DS_mini_project\ds_mini-project\Dataset\billings.csv")
renewal_calls = pd.read_csv(r"D:\INT230\DS_mini_project\ds_mini-project\Dataset\renewal_calls.csv")
emails = pd.read_csv(r"D:\INT230\DS_mini_project\ds_mini-project\Dataset\emails.csv")
cc_calls = pd.read_csv(r"D:\INT230\DS_mini_project\ds_mini-project\Dataset\cc_calls.csv")

C:\Users\nehas\AppData\Local\Temp\ipykernel_8240\3282648463.py:1: DtypeWarning: Columns (0: Proforma_Auto_Renewal, 1: Proforma_World_Pay_Token, 2: Current_Anchor_List, 3: Last_Renewal, 4: Last_Band) have mixed types. Specify dtype option on import or set low_memory=False.
  billings = pd.read_csv(r"D:\INT230\DS_mini_project\ds_mini-project\Dataset\billings.csv")
C:\Users\nehas\AppData\Local\Temp\ipykernel_8240\3282648463.py:2: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitor

3. Basic inspection function

In [274]:
def inspect(df, name):
    print(f"\n{name} Dataset")
    print("-" * 40)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nDuplicate Rows:", df.duplicated().sum())

inspect(billings, "Billings")
inspect(renewal_calls, "Renewal Calls")
inspect(emails, "Emails")
inspect(cc_calls, "CC Calls")


Billings Dataset
----------------------------------------
Shape: (122082, 59)

Columns:
['Co_Ref', 'Renewal_Month', 'Connection_Net', 'Connection_Qty', 'Discount_Amount', 'Sustainability_Score', 'Total_Renewal_Score_New', 'Starting_Connection_Net', 'Starting_Connection_Qty', 'Last_Years_Price', 'Last_Years_Date_Paid', 'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores', 'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 'Proforma_Date', 'Current_Anchorings', 'Current_Anchor_List', 'Payment_Timeframe', 'Registration_Date', 'Proforma_Account_Stage', 'Proforma_Audit_Status', 'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token', 'Renewal_Score_At_Release', 'Proforma_Membership_Status', 'Proforma_Approved_Lists', 'Tenure_Years', 'Band', 'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status', 'Starting_Net', 'Starting_Vat', 'Starting_Gross', 'Starting_Membership_Net', 'Starting_Package_Net', 'Starting_PQQ_Net', 'Gross', 'Membership_Net', 'Package_Net', 'PQQNet',

In [275]:
billings.head()

,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,1,3,1.0,01-11-2023,Band B,664.0,1.0,1,2024,01-11-2024
1,VD3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,1,1,1.0,NaN,NaN,NaN,NaN,1,2025,01-08-2025
2,DV8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
3,EZ9894,01-06-2025,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,1,4+,1.0,01-06-2024,Band C1,749.0,1.0,1,2025,01-06-2025
4,FA8957,01-03-2025,NaN,NaN,NaN,9.5,42.5,NaN,NaN,799.0,...,1,3,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025


In [276]:
renewal_calls.head()


,Call_ID,Call_Direction,Co_Ref,Call_Date,Churn_Category,Complaint_Category,Customer_Reaction_Category,Agent_Renewal_Pitch_Category,Customer_Renewal_Response_Category,Agent_Response_Category,...,Customer_Response,Desire_To_Cancel,Discount_Offered,Justification_Category,Reason_For_Renewal_Category,Agent_Response_To_Cancel_Category,Argument_That_Convinced_Customer_to_Stay_Category,Analysed_Call,Call_Number,Call_Year
0,5.950000e+11,Outbound,UB0899,29-01-2025,NaN,NaN,Not Mentioned,Discussion / Introduction / Inquiry,Discount and Offer,Discount and Offer,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,3,2025
1,5.970000e+11,OUT_BOUND,HN5141,26-02-2025,NaN,NaN,NaN,Price and Cost,Agreement,Customer Communication,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,2,2025
2,5.950000e+11,Outbound,BP5009,24-01-2025,NaN,NaN,NaN,Expiration / Due,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
3,6.520000e+11,OUT_BOUND,XP8119,09-06-2025,NaN,NaN,NaN,Auto / Automatic,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
4,5.370000e+11,Outbound,ZL7978,20-08-2024,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,28,2024


In [277]:
emails.head()

,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


In [278]:
cc_calls.head()

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,08-05-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,25-11-2024,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,23-10-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,13-01-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,19-03-2025,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025


Unique values

In [279]:
billings['Prospect_Outcome'].value_counts()

Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64

In [280]:
emails['Time_to_Renewal'].value_counts()

Time_to_Renewal
prior_year     40022
14_out         32493
45_out         28008
pre_renewal    22866
Name: count, dtype: int64

Check date columns

In [281]:
billings['Prospect_Renewal_Date'].head()

0    05-11-2024
1    09-08-2025
2    12-03-2025
3    29-06-2025
4    25-03-2025
Name: Prospect_Renewal_Date, dtype: str

In [282]:
renewal_calls['Call_Date'].head()

0    29-01-2025
1    26-02-2025
2    24-01-2025
3    09-06-2025
4    20-08-2024
Name: Call_Date, dtype: str

In [283]:
cc_calls['Call_Date'].head()

0    08-05-2025
1    25-11-2024
2    23-10-2024
3    13-01-2025
4    19-03-2025
Name: Call_Date, dtype: str

### 3. Initial Data Cleaning

In this step, we perform basic data cleaning to prepare the datasets for analysis.

The objectives of this step are:

* To correct data types (especially date columns)
* To remove duplicate records
* To handle obvious missing values
* To standardize categorical values
* To drop irrelevant or junk columns

This step ensures that the data is in a usable format for further analysis and feature engineering.


1. Copy datasets

In [284]:
billings_clean = billings.copy()
renewal_calls_clean = renewal_calls.copy()
emails_clean = emails.copy()
cc_calls_clean = cc_calls.copy()

2. Convert date columns

In [285]:
billings_clean['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_clean['Prospect_Renewal_Date'],
    dayfirst=True,
    errors='coerce'
)

renewal_calls_clean['Call_Date'] = pd.to_datetime(
    renewal_calls_clean['Call_Date'],
    dayfirst=True,
    errors='coerce'
)

cc_calls_clean['Call_Date'] = pd.to_datetime(
    cc_calls_clean['Call_Date'],
    dayfirst=True,
    errors='coerce'
)

3. Remove duplicates

In [286]:
billings_clean = billings_clean.drop_duplicates()
renewal_calls_clean = renewal_calls_clean.drop_duplicates()
emails_clean = emails_clean.drop_duplicates()
cc_calls_clean = cc_calls_clean.drop_duplicates()

4. Drop obvious junk columns

In [287]:
# Example: unnamed columns
renewal_calls_clean = renewal_calls_clean.loc[:, ~renewal_calls_clean.columns.str.contains('^Unnamed')]
cc_calls_clean = cc_calls_clean.loc[:, ~cc_calls_clean.columns.str.contains('^Unnamed')]

5. Standardize text

In [288]:
def clean_text(df):
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = df[col].apply(
            lambda x: x.strip().lower() if isinstance(x, str) else x
        )
    return df

# Clean target column properly

billings_clean = clean_text(billings_clean)
renewal_calls_clean = clean_text(renewal_calls_clean)
emails_clean = clean_text(emails_clean)
cc_calls_clean = clean_text(cc_calls_clean)

6. Handle obvious missing values

In [289]:
# Drop completely empty columns (all datasets)
billings_clean = billings_clean.dropna(axis=1, how='all')
renewal_calls_clean = renewal_calls_clean.dropna(axis=1, how='all')
emails_clean = emails_clean.dropna(axis=1, how='all')
cc_calls_clean = cc_calls_clean.dropna(axis=1, how='all')


# BILLINGS

# Drop useless column (100% missing)
if 'Last_Years_Date_Paid' in billings_clean.columns:
    billings_clean = billings_clean.drop(columns=['Last_Years_Date_Paid'])

# Drop rows with critical missing values

billings_clean = billings_clean.dropna(
    subset=['Prospect_Outcome','Prospect_Renewal_Date', 'Co_Ref']
)
# RENEWAL CALLS

# Drop rows where Co_Ref is missing (cannot join later)
renewal_calls_clean = renewal_calls_clean.dropna(subset=['Co_Ref'])


# EMAILS

# Drop rows where Co_Ref is missing
emails_clean = emails_clean.dropna(subset=['Co_Ref'])


# CC CALLS

# Drop rows where Call_Date or Co_Ref is missing
cc_calls_clean = cc_calls_clean.dropna(subset=['Call_Date', 'Co_Ref'])


print("After Cleaning:\n")
print("Billings Shape:", billings_clean.shape)
print("Renewal Calls Shape:", renewal_calls_clean.shape)
print("Emails Shape:", emails_clean.shape)
print("CC Calls Shape:", cc_calls_clean.shape)

After Cleaning:

Billings Shape: (122082, 58)
Renewal Calls Shape: (153269, 40)
Emails Shape: (123389, 27)
CC Calls Shape: (31636, 33)


### STEP 4:Target Definition & Cutoff Logic

In this step, we define the target variable (churn) and create a cutoff date to enforce time-based data usage.

1. Check Target Distribution

This step helps us understand the distribution of the target variable (Prospect_Outcome).

It allows us to:

* Identify different outcome categories
* Check class imbalance
* Detect unwanted categories (e.g., "Open")

In [290]:
billings_clean['Prospect_Outcome'].value_counts()

Prospect_Outcome
won        101226
churned     12668
open         8188
Name: count, dtype: int64

2. Remove "Open" Outcomes

Rows with "Open" status are removed because they do not represent a final outcome.

"Won" → Customer renewed
"Churned" → Customer did not renew
"Open" → Outcome not finalized (cannot be used for training)

In [291]:
billings_clean = billings_clean[
    billings_clean['Prospect_Outcome'].isin(['won', 'churned'])
]

3. Create Target Variable (churn)

We convert the categorical outcome into a numerical target variable:

churn = 1 → Customer churned
churn = 0 → Customer renewed

This transformation is required for machine learning models.

In [292]:
billings_clean['churn'] = billings_clean['Prospect_Outcome'].map({
    'won': 0,
    'churned': 1
})

4. Verify Target Variable

This step ensures that the target variable has been created correctly.

It also helps confirm:

Distribution of churn vs non-churn
No unexpected values

In [293]:
billings_clean['churn'].value_counts()

churn
0    101226
1     12668
Name: count, dtype: int64

5. Create Cutoff Date

A cutoff date is created to enforce time-based feature extraction.

Only data available before 45 days of the renewal date is allowed

Formula:
cutoff_date = Prospect_Renewal_Date - 45 days

In [294]:
billings_clean['cutoff_date'] = (
    billings_clean['Prospect_Renewal_Date'] - pd.Timedelta(days=45)
)

6. Validate Cutoff Date

This step verifies that the cutoff date has been calculated correctly.



In [295]:
billings_clean[['Prospect_Renewal_Date', 'cutoff_date']].head()

,Prospect_Renewal_Date,cutoff_date
0,2024-11-05,2024-09-21
1,2025-08-09,2025-06-25
2,2025-03-12,2025-01-26
3,2025-06-29,2025-05-15
4,2025-03-25,2025-02-08


### STEP 5 : Build Modeling Dataset 

In this step, we construct the final modeling dataset by transforming raw interaction data into structured features.

The goal is to:

* Aggregate data at the customer level
* Ensure no data leakage using cutoff_date
* Combine all datasets into a single table

1. Create Base Dataset

We start by copying the cleaned billing dataset.

This dataset acts as the base table because:

* It contains the target variable (churn)
* It defines the unit of analysis (1 customer × 1 renewal)
* All other datasets will be merged onto this

In [296]:
df = billings_clean.copy()

2. Filter Renewal Calls Using Cutoff Date

We merge renewal calls with the base dataset to bring in the cutoff_date.

Then we filter the data to keep only calls that happened before the cutoff_date.

This ensures:

* No future data is used
* Model mimics real-world prediction

In [297]:
renewal_calls_filtered = renewal_calls_clean.merge(
    df[['Co_Ref', 'cutoff_date']],
    on='Co_Ref',
    how='inner'
)

renewal_calls_filtered = renewal_calls_filtered[
    renewal_calls_filtered['Call_Date'] <= renewal_calls_filtered['cutoff_date']
]

3. Create Renewal Call Features

We aggregate the filtered renewal calls at the customer level.

This converts multiple call records into a single feature:

* total_renewal_calls → number of renewal interactions

This is important because models require one row per customer.

In [298]:
renewal_features = renewal_calls_filtered.groupby(
    ['Co_Ref', 'cutoff_date']
).agg(
    total_renewal_calls=('Call_Date', 'count')
).reset_index()

4. Filter Customer Care Calls

We apply the same logic to customer care calls:

* Merge with cutoff_date
* Keep only calls before cutoff

This dataset captures customer issues and complaints before renewal.

In [299]:
cc_calls_filtered = cc_calls_clean.merge( 
    df[['Co_Ref', 'cutoff_date']], on='Co_Ref', how='inner' ) 
cc_calls_filtered = cc_calls_filtered[ cc_calls_filtered['Call_Date'] <= cc_calls_filtered['cutoff_date'] ]

5. Create Customer Care Features

We aggregate customer care calls to create features such as:

* total_cc_calls → number of support interactions

These features help capture customer dissatisfaction.

In [300]:
cc_features = cc_calls_filtered.groupby(
    ['Co_Ref', 'cutoff_date']
).agg(
    total_cc_calls=('Call_Date', 'count')
).reset_index()

6. Filter Emails Dataset

The emails dataset is already grouped by time buckets.

We select only relevant buckets:

* prior_year
* 45_out

These represent interactions before the cutoff date.

In [301]:
emails_filtered = emails_clean[
    emails_clean['Time_to_Renewal'].isin(['prior_year', '45_out'])
]

7. Create Email Features

We aggregate email interactions to create:

* total_emails → number of CRM interactions

This helps capture engagement level of the customer.

In [302]:
email_features = emails_filtered.groupby(
    ['Co_Ref']
).agg(
    total_emails=('Time_to_Renewal', 'count')
).reset_index()

8. Merge All Features

We combine all generated features with the base dataset.

Each dataset contributes additional information about the customer.

In [303]:
df = df.merge(renewal_features, on=['Co_Ref', 'cutoff_date'], how='left')
df = df.merge(cc_features, on=['Co_Ref', 'cutoff_date'], how='left')
df = df.merge(email_features, on='Co_Ref', how='left')

9. Handle Missing Values After Merge

After merging, some customers may not have interactions.

We replace missing values with 0:

* No calls → 0
* No emails → 0


In [304]:
df[['total_renewal_calls', 'total_cc_calls', 'total_emails']] = df[
    ['total_renewal_calls', 'total_cc_calls', 'total_emails']
].fillna(0)

10. Final Dataset Check

We inspect the final dataset to ensure:

* Correct shape
* Features are created properly
* No unexpected issues


In [ ]:
df.head(10)


,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out,churn,cutoff_date,total_renewal_calls,total_cc_calls,total_emails
0,vt6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,664.0,1.0,1,2024,01-11-2024,0,2024-09-21,0.0,0.0,1.0
1,vd3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,NaN,NaN,1,2025,01-08-2025,0,2025-06-25,0.0,0.0,1.0
2,dv8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,749.0,1.0,1,2025,01-03-2025,0,2025-01-26,0.0,0.0,0.0
3,ez9894,01-06-2025,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,749.0,1.0,1,2025,01-06-2025,0,2025-05-15,1.0,5.0,2.0
4,fa8957,01-03-2025,NaN,NaN,NaN,9.5,42.5,NaN,NaN,799.0,...,749.0,1.0,1,2025,01-03-2025,0,2025-02-08,4.0,2.0,4.0
5,qs2598,01-06-2025,NaN,NaN,NaN,8.0,40.5,NaN,NaN,799.0,...,749.0,1.0,1,2025,01-06-2025,0,2025-05-08,1.0,0.0,1.0
6,cj9355,01-03-2025,NaN,NaN,NaN,8.0,43.0,NaN,NaN,799.0,...,749.0,1.0,1,2025,01-03-2025,0,2025-01-31,0.0,0.0,2.0
7,fp1608,01-11-2024,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,749.0,1.0,1,2024,01-11-2024,0,2024-09-24,0.0,0.0,2.0
8,tz9717,01-11-2024,NaN,NaN,NaN,8.0,41.0,NaN,NaN,799.0,...,749.0,1.0,1,2024,01-11-2024,0,2024-10-06,0.0,0.0,1.0
9,up7099,01-04-2025,NaN,NaN,NaN,8.0,42.0,NaN,NaN,799.0,...,749.0,3.0,1,2025,01-04-2025,0,2025-02-18,6.0,1.0,2.0


In [307]:
df.shape

(113894, 63)

In [310]:
df.to_csv("model_dataset_option_A.csv", index=False)